In [10]:
from pathlib import Path
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Tratamento inicial dos dados

## Para que serve este notebook

Antes de calcular qualquer indicador, é preciso **arrumar a casa**. Os dados chegam de dois sistemas diferentes da fábrica:

- **MES** (o sistema usado pela Produção): é o "diário de bordo" da linha. Cada vez que um equipamento para, o operador anota quando parou, quando voltou, qual equipamento e qual componente estavam envolvidos.
- **SAP** (o sistema usado pela Manutenção): é a "agenda de serviços" da equipe de manutenção. Cada serviço vira uma **ordem**, com número, tipo, um texto curto descrevendo o problema e o horário de início e fim do trabalho.

Os dois foram preenchidos à mão, por pessoas diferentes, com pressa. Então aparecem erros de digitação, acentos diferentes, registros repetidos e horários impossíveis. Este notebook faz a **faxina**: padroniza, junta o que é a mesma coisa e marca o que não pode ser usado.

## Como marcamos a qualidade de cada linha: "carimbo vermelho" e "post-it amarelo"

Em vez de apagar linhas ruins (e perder o rastro delas), cada linha recebe duas marcações:

- **`valido`** é o **carimbo vermelho**. Vale `False` só quando o dado está **errado de verdade**: um horário impossível (o reparo terminou antes de começar), um registro duplicado ou uma informação essencial vazia. O motivo fica escrito em `motivo_invalido`. Linha inválida não entra nas contas.
- **`alertas`** é o **post-it amarelo**. O registro é aproveitável, mas tem algo estranho que vale a pena saber (por exemplo, uma ordem com mais de 24 h). Ele **continua nas contas**; o post-it só avisa quem for fazer uma análise mais detalhada.

**Por que essa decisão:** apagar dados é irreversível e esconde o tamanho do problema. Marcar permite contar quantos registros tinham defeito (o que já é um resultado para a fábrica) e, se alguém discordar de uma regra, basta mudar a regra e rodar de novo.

### Ferramentas de limpeza usadas daqui para frente

A célula abaixo só **define ferramentas** (funções) que serão usadas várias vezes. Nenhum dado é alterado ainda. O que cada uma faz, em linguagem simples:

| Função | O que faz | Exemplo / analogia |
|---|---|---|
| `to_nfc` | Padroniza a forma como o computador guarda letras acentuadas e tira espaços sobrando nas pontas. | Para o computador, "É" pode ser guardado como **uma** letra ou como "E + acento" (duas peças). Na tela parecem iguais, mas o computador acha que são palavras diferentes. Essa função deixa todas no mesmo formato. |
| `sem_acento` | Remove os acentos. | `TRANSFERÊNCIA` vira `TRANSFERENCIA`. Assim, quem digitou com e sem acento passa a escrever "a mesma palavra". |
| `clean_cat` | Padroniza uma categoria (nome de máquina, equipamento, componente): sem acento, tudo em maiúsculas, um espaço só entre palavras, e campo vazio vira "vazio" de verdade. | `" Aquecimento  "` e `"AQUECIMENTO"` passam a ser o mesmo texto. |
| `to_serial_excel` | Converte datas que o Excel guardou como número. | O Excel conta os dias a partir de 30/12/1899. O número `46023` é, na verdade, uma data de 2026. |
| `add_motivo` | Vai anotando os motivos de invalidez/alerta sem apagar o que já estava anotado. | Como acrescentar uma linha numa ficha, sem rasurar as anteriores. |

In [11]:
# Importação do arquivo
FILE = Path("./data/raw/base.xlsx").resolve()


def to_nfc(text):
    if pd.isna(text):
        return pd.NA
    return unicodedata.normalize("NFC", str(text)).strip()

def sem_acento(text):
    if pd.isna(text):
        return pd.NA
    decomposto = unicodedata.normalize("NFKD", str(text))
    return "".join(c for c in decomposto if not unicodedata.combining(c))

def clean_cat(serie):
    s = serie.map(sem_acento).astype("string").str.strip().str.upper()
    return s.str.replace(r"\s+", " ", regex=True).replace("", pd.NA)

def to_serial_excel(valor):
    return pd.to_datetime(valor, unit="D", origin="1899-12-30")

def add_motivo(serie_atual, condicao, texto):
    cond = pd.Series(condicao, index=serie_atual.index).fillna(False).astype(bool)
    resultado = np.where(cond,
        np.where(serie_atual == "", texto, serie_atual + " | " + texto),
        serie_atual)
    return pd.Series(resultado, index=serie_atual.index)

def fim_max_anterior(df, chave):
    return df.groupby(chave)["fim"].transform(lambda s: s.cummax().shift())

def uniao_horas(df, chave):
    m = fim_max_anterior(df, chave)
    inicio_efetivo = df["inicio"].where(m.isna() | (df["inicio"] >= m), m)
    contrib = ((df["fim"] - inicio_efetivo).dt.total_seconds() / 3600).clip(lower=0)
    return contrib.groupby(df[chave]).sum()

## Tratamento do MES (o diário de bordo da Produção)

Na próxima célula lemos a aba do MES, damos nomes simples às colunas (sem acento e sem espaço, para facilitar o código), padronizamos os textos e criamos um número de identificação (`id_registro`) para cada linha.

**`id_registro`:** é a "etiqueta de rastreio" da linha. Qualquer número calculado lá na frente (por exemplo, "a máquina A parou 606 h") pode ser rastreado até as linhas originais da planilha. Isso dá confiança: se alguém da fábrica duvidar de um número, dá para mostrar de onde ele veio.

**`data_calendario`:** o turno 3 vai das 22h às 06h. Uma parada às 2h da manhã do dia 5 é registrada pela Produção como "turno 3 do dia 4". Guardamos as duas datas: a do turno (como a fábrica fala) e a do calendário (o dia real em que aconteceu).

In [12]:
mes = pd.read_excel(FILE, sheet_name="MES - PRODUÇÃO")

# Normaliza Unicode dos nomes de coluna
mes.columns = [to_nfc(c) for c in mes.columns]

# Renomeia colunas para facilitar o uso
mes = mes.rename(columns={
    "MÁQUINA": "maquina",
    "Data": "data_turno",
    "Famila": "familia",
    "EQUIPAMENTO": "equipamento",
    "Componente": "componente",
    "Tur": "turno",
    "Inicio": "inicio",
    "Término": "fim",
    "Horas": "horas",
})

# Id para rastrear direto para a linha original do Excel
mes["id_registro"] = range(1, len(mes) + 1)

# Padroniza categorias (sem acento, caixa alta, espaços simples)
for c in ["maquina", "familia", "equipamento", "componente"]:
    mes[c] = clean_cat(mes[c])

# Data de calendário (a data_turno do turno 3 é a do dia anterior após a meia-noite)
mes["data_calendario"] = mes["inicio"].dt.normalize()

mes.head()

,maquina,data_turno,familia,equipamento,componente,turno,inicio,fim,horas,id_registro,data_calendario
0,A,2026-01-02,AJUSTE,DISCO,BCL,2,2026-01-02 18:33:00,2026-01-02 19:02:00,0.4833,1,2026-01-02
1,A,2026-01-02,QUEBRA,ROLOS,FD,1,2026-01-02 08:30:00,2026-01-02 09:03:00,0.5500,2,2026-01-02
2,A,2026-01-02,AJUSTE,DISCO,BCL,2,2026-01-02 14:37:00,2026-01-02 15:03:00,0.4333,3,2026-01-02
3,A,2026-01-03,AJUSTE,DISCO,DI,3,2026-01-04 02:20:00,2026-01-04 02:42:00,0.3667,4,2026-01-04
4,A,2026-01-03,AJUSTE,DISCO,BCL,3,2026-01-03 23:06:00,2026-01-03 23:31:00,0.4167,5,2026-01-03


### Agrupamento dos registros em paradas

**O problema:** uma única parada do equipamento pode aparecer **picotada** em vários registros no MES. Dois casos comuns:

1. **Troca de turno:** a parada começa às 21:33 e só termina às 22:36. O operador do turno 2 fecha o registro às 21:59, e o do turno 3 abre outro às 22:00. São dois registros, mas foi **uma** parada.
2. **Serviços simultâneos:** dois componentes do mesmo equipamento são consertados ao mesmo tempo. São dois registros (um por componente), mas a máquina parou **uma** vez.

**A regra:** registros do mesmo equipamento que **se sobrepõem** no tempo, ou que estão separados por **até `GAP_MAX_MIN` = 2 minutos**, são "costurados" numa única parada, que recebe um número próprio (`id_parada`).

Registros com horário impossível (sem início ou fim, ou com fim antes do início) não são costurados a nada e ficam sem `id_parada`.

In [13]:
GAP_MAX_MIN = 2

mes = mes.drop(columns=["gap_anterior_min", "id_parada"], errors="ignore")
mes = mes.sort_values(["maquina", "equipamento", "inicio", "fim"]).reset_index(drop=True)

tempo_ok = mes["inicio"].notna() & mes["fim"].notna() & (mes["fim"] > mes["inicio"])

v = mes.loc[tempo_ok].assign(_eq=mes["equipamento"].fillna("SEM_EQUIPAMENTO"))
grupo = ["maquina", "_eq"]

# Gap até o MAIOR fim anterior do equipamento (negativo = sobreposição)
gap_min = (v["inicio"] - fim_max_anterior(v, grupo)).dt.total_seconds() / 60

# Nova parada só quando o gap passa do limite; sobreposição (gap < 0) continua a mesma parada
nova = gap_min.isna() | (gap_min > GAP_MAX_MIN)
n_parada = nova.groupby([v["maquina"], v["_eq"]]).cumsum().astype(int)

mes["gap_anterior_min"] = gap_min.round(1)
mes["id_parada"] = (v["maquina"] + "-" + v["_eq"].str.replace(" ", "_") + "-"
                    + n_parada.astype(str).str.zfill(4))

mes.head()

,maquina,data_turno,familia,equipamento,componente,turno,inicio,fim,horas,id_registro,data_calendario,gap_anterior_min,id_parada
0,A,2026-01-03,AJUSTE,AQUECIMENTO,BDR,2,2026-01-03 21:33:00,2026-01-03 21:59:00,0.4333,7,2026-01-03,NaN,A-AQUECIMENTO-0001
1,A,2026-01-03,AJUSTE,AQUECIMENTO,BDR,3,2026-01-03 22:00:00,2026-01-03 22:36:00,0.6000,8,2026-01-03,1.0,A-AQUECIMENTO-0001
2,A,2026-01-03,AJUSTE,AQUECIMENTO,BDC,3,2026-01-04 04:01:00,2026-01-04 04:20:00,0.3167,9,2026-01-04,325.0,A-AQUECIMENTO-0002
3,A,2026-01-04,QUEBRA,AQUECIMENTO,CLH,1,2026-01-04 07:25:00,2026-01-04 07:54:00,0.4833,17,2026-01-04,185.0,A-AQUECIMENTO-0003
4,A,2026-01-07,QUEBRA,AQUECIMENTO,FQCO,3,2026-01-07 22:01:00,2026-01-08 01:21:00,3.3333,34,2026-01-07,5167.0,A-AQUECIMENTO-0004


### Validade e alertas por registro do MES

Aqui aplicamos o "carimbo vermelho" (inválido) e o "post-it amarelo" (alerta) explicados no início.

| Regra | Efeito | Por quê, em palavras simples |
|---|---|---|
| sem início ou fim; fim antes do início; duração zero | **inválido** | É um tempo impossível: não existe parada que termina antes de começar. |
| sobrepõe outro registro do **mesmo** componente | **inválido** (duplicata) | O mesmo componente não pode quebrar duas vezes ao mesmo tempo. Se aparece duas vezes no mesmo horário, alguém digitou duas vezes. Mantemos o primeiro e marcamos o segundo. |
| componente ou equipamento vazio | **inválido** | A parada pode ter acontecido, mas sem saber **o que** quebrou ela não serve para as análises por componente, que são o foco do trabalho. |
| coluna `horas` diferente de (fim − início) | alerta | A planilha traz uma coluna de horas já calculada, que às vezes não bate com os horários. Confiamos nos horários (fim − início), porque são eles que entram na "união" do tempo parado. |

In [14]:
mes = mes.drop(columns=["valido", "motivo_invalido", "alertas",
                        "confiavel", "motivo_desconfianca"], errors="ignore")

invalido = pd.Series("", index=mes.index)
alerta = pd.Series("", index=mes.index)

dur_calc_h = (mes["fim"] - mes["inicio"]).dt.total_seconds() / 3600

# Inválidos
invalido = add_motivo(invalido, mes["inicio"].isna() | mes["fim"].isna(), "sem início ou fim")
invalido = add_motivo(invalido, dur_calc_h < 0, "fim antes do início")
invalido = add_motivo(invalido, dur_calc_h == 0, "duração zero")
invalido = add_motivo(invalido, mes["componente"].isna(), "componente vazio")
invalido = add_motivo(invalido, mes["equipamento"].isna(), "equipamento vazio")

# Mesmo equipamento E mesmo componente sobrepostos -> duplicata (mantém o primeiro)
o = (mes.loc[tempo_ok & mes["componente"].notna()]
        .assign(_eq=mes["equipamento"].fillna("SEM_EQUIPAMENTO"))
        .sort_values(["maquina", "_eq", "componente", "inicio", "fim"]))
dup = o["inicio"] < fim_max_anterior(o, ["maquina", "_eq", "componente"])
invalido = add_motivo(invalido, dup.reindex(mes.index, fill_value=False),
                      "duplicata: sobrepõe registro do mesmo componente")

# Alertas
alerta = add_motivo(alerta, (dur_calc_h - mes["horas"]).abs() > 1/60, "horas ≠ fim − início (usado fim − início)")

mes["valido"] = invalido.eq("")
mes["motivo_invalido"] = invalido
mes["alertas"] = alerta

print(mes["valido"].value_counts())
print(mes.loc[~mes["valido"], "motivo_invalido"].value_counts())

valido
True     1613
False      27
Name: count, dtype: int64
motivo_invalido
componente vazio                                    19
duplicata: sobrepõe registro do mesmo componente     8
Name: count, dtype: int64


**O que a saída mostra:** de 1.640 registros, 27 (≈ 1,6%) foram marcados como inválidos: 19 por componente vazio e 8 por duplicata. É pouca coisa, o que indica que o MES, apesar de manual, é razoavelmente bem preenchido.

### Tabela de paradas (uma linha por `id_parada`)

Agora que os registros foram "costurados" em paradas, montamos uma tabela com **uma linha por parada**. Ela é a base de todos os indicadores dos próximos notebooks. Colunas principais:

- **`duracao_h`**: tempo real em que o equipamento ficou parado, calculado pela **união** dos registros válidos (o tempo sobreposto conta uma vez só, como explicado na régua pintada).
- **`soma_registros_h`**: a soma simples das durações, guardada só para comparação. A diferença entre as duas (`tempo_sobreposto_h`) mostra quanto a conta "ingênua" inflaria o tempo parado.
- **Família e componente dominantes**: quando uma parada tem vários registros, qual deles "representa" a parada? Escolhemos o **registro de maior duração**: foi o problema que mais segurou a máquina parada. A coluna `contem_quebra` avisa se **algum** registro da parada foi uma QUEBRA, mesmo que não seja o dominante.
- **Micro-parada**: parada inteira com **menos de `MICRO_PARADA_MIN` = 2 minutos**. Não é erro de dado: é uma parada real, mas tão curta que não representa um conserto (provavelmente um ajuste rápido do operador). Ela é separada para não "puxar para baixo" o tempo médio de reparo nos próximos notebooks.
- **`ids_registro`**: lista das linhas originais que formam a parada, para rastrear qualquer número até a planilha.

In [15]:
MICRO_PARADA_MIN = 2

rv = (mes.loc[mes["valido"]]
         .sort_values(["id_parada", "inicio", "fim"])
         .assign(_dur_h=dur_calc_h))

paradas = rv.groupby("id_parada").agg(
    maquina=("maquina", "first"),
    equipamento=("equipamento", "first"),
    inicio=("inicio", "min"),
    fim=("fim", "max"),
    soma_registros_h=("_dur_h", "sum"),
    n_registros=("id_registro", "size"),
    n_componentes=("componente", "nunique"),
    contem_quebra=("familia", lambda s: s.eq("QUEBRA").any()),
    tem_componente_vazio=("componente", lambda s: s.isna().any()),
    ids_registro=("id_registro", lambda s: ",".join(map(str, s))),
)

paradas["duracao_h"] = uniao_horas(rv, "id_parada")
paradas["tempo_sobreposto_h"] = (paradas["soma_registros_h"] - paradas["duracao_h"]).round(4)

# Dominantes = registro válido de maior duração (componente: ignora vazios)
maior = rv.sort_values("_dur_h", ascending=False)
paradas["familia_dominante"] = maior.drop_duplicates("id_parada").set_index("id_parada")["familia"]
paradas["componente_dominante"] = (maior.dropna(subset=["componente"])
                                        .drop_duplicates("id_parada").set_index("id_parada")["componente"])

paradas["micro_parada"] = (paradas["duracao_h"] * 60).round(1) < MICRO_PARADA_MIN
paradas["n_registros_descartados"] = (mes.loc[~mes["valido"]].groupby("id_parada").size()
                                         .reindex(paradas.index, fill_value=0))

paradas = paradas.reset_index()[[
    "id_parada", "maquina", "equipamento", "inicio", "fim", "duracao_h",
    "soma_registros_h", "tempo_sobreposto_h", "familia_dominante", "componente_dominante",
    "contem_quebra", "n_registros", "n_componentes", "tem_componente_vazio",
    "micro_parada", "n_registros_descartados", "ids_registro",
]]

mes = mes.drop(columns=["gap_anterior_min"])

print(f"{len(mes)} registros -> {len(paradas)} paradas")
print(f"Horas: soma ingênua = {paradas['soma_registros_h'].sum():.1f} h | "
      f"união = {paradas['duracao_h'].sum():.1f} h")
print(f"total de descartados: {mes.loc[~mes['valido'], 'id_registro'].nunique()} registros")
paradas.head()

1640 registros -> 1501 paradas
Horas: soma ingênua = 1969.8 h | união = 1956.6 h
total de descartados: 27 registros


,id_parada,maquina,equipamento,inicio,fim,duracao_h,soma_registros_h,tempo_sobreposto_h,familia_dominante,componente_dominante,contem_quebra,n_registros,n_componentes,tem_componente_vazio,micro_parada,n_registros_descartados,ids_registro
0,A-AQUECIMENTO-0001,A,AQUECIMENTO,2026-01-03 21:33:00,2026-01-03 22:36:00,1.033333,1.033333,0.0,AJUSTE,BDR,False,2,1,False,False,0,"7,8"
1,A-AQUECIMENTO-0002,A,AQUECIMENTO,2026-01-04 04:01:00,2026-01-04 04:20:00,0.316667,0.316667,0.0,AJUSTE,BDC,False,1,1,False,False,0,9
2,A-AQUECIMENTO-0003,A,AQUECIMENTO,2026-01-04 07:25:00,2026-01-04 07:54:00,0.483333,0.483333,0.0,QUEBRA,CLH,True,1,1,False,False,0,17
3,A-AQUECIMENTO-0004,A,AQUECIMENTO,2026-01-07 22:01:00,2026-01-08 01:21:00,3.333333,3.333333,0.0,QUEBRA,FQCO,True,1,1,False,False,0,34
4,A-AQUECIMENTO-0005,A,AQUECIMENTO,2026-01-08 09:22:00,2026-01-08 09:57:00,0.583333,0.583333,0.0,QUEBRA,CLH,True,1,1,False,False,0,35


**O que a saída mostra:** os 1.640 registros viraram **1.501 paradas**. Somando os registros de forma ingênua daria 1.969,8 h; pela união, o tempo parado real é 1.956,6 h. A diferença (≈ 13 h) é pequena, mas mostra que a costura evita contar duas vezes o mesmo tempo.

## Tratamento do SAP (a agenda de serviços da Manutenção)

No SAP, cada linha é uma **ordem de manutenção**: um serviço aberto pela equipe, com número, tipo (corretiva, corretiva planejada...), local da máquina, um texto curto ("PROBLEMA NO QMD") e os horários de início e fim do trabalho.

### De onde sai a máquina de cada ordem

No SAP, o "endereço" do equipamento vem num código chamado **localização técnica**, escrito em níveis separados por ponto, como um endereço postal:

`9001 . FABX . CEL01 . MAQA . FRN01` → planta . fábrica . célula . **máquina** . **subconjunto**

Pegamos o pedaço da posição da máquina (`NIVEL_MAQUINA`) e traduzimos com uma **tabela de-para** (`MAPA_MAQUINA_SAP`): `MAQA → A`, `MAQB → B`, etc.


**Por que a coluna `Descrição` do SAP vira `descricao_equip` e não `equipamento`:** os dois sistemas usam **vocabulários diferentes** para a mesma máquina. O SAP fala "FORNO 1" e "LONAS"; o MES fala "AQUECIMENTO" e "DISCO". Dar o mesmo nome às duas colunas faria parecer que são a mesma coisa. Descobrir qual palavra de um corresponde a qual do outro é justamente o trabalho do notebook de conciliação.

Também criamos duas versões do texto: `texto_original` (exatamente como o mantenedor escreveu) e `texto_padronizado` (sem acento, maiúsculas, sem ponto final), que é a versão usada para comparações.

In [16]:
NIVEL_MAQUINA = 3        # 9001.FABX.CEL01.[MAQA].FRN01  (índice a partir de 0)
NIVEL_SUBCONJUNTO = 4    # 9001.FABX.CEL01.MAQA.[FRN01]

# Tabela de-para: código da localização técnica no SAP -> máquina no MES
MAPA_MAQUINA_SAP = {
    "MAQA": "A",
    "MAQB": "B",
    "MAQC": "C",
    "MAQD": "D",
}

sap = pd.read_excel(FILE, sheet_name="SAP - MANUTENÇÃO",
                    dtype={"InícReal hr.": str, "Fim real": str})

sap.columns = [to_nfc(c) for c in sap.columns]

sap = sap.rename(columns={
    "N. Ordem": "n_ordem",
    "Tipo": "tipo",
    "Data-base iníc.": "data_base",
    "Texto breve": "texto_original",
    "Loc.instalação": "local_instalacao",
    "Descrição": "descricao_equip",
    "Dta.início real": "_d_ini",
    "InícReal hr.": "_h_ini",
    "Dt.real fim": "_d_fim",
    "Fim real": "_h_fim",
    "Tip.ativ.manut.": "tipo_atividade",
})

for c in ["tipo", "descricao_equip", "tipo_atividade", "local_instalacao"]:
    sap[c] = clean_cat(sap[c])

# texto_original: só normalização Unicode (preserva o que o mantenedor escreveu; vazio continua vazio)
# texto_padronizado: sem acento, caixa alta, espaços simples, sem ponto final -> base da classificação
sap["texto_original"] = sap["texto_original"].map(to_nfc).astype("string")
sap["texto_padronizado"] = clean_cat(sap["texto_original"]).str.rstrip(". ").replace("", pd.NA)
sap["n_palavras"] = sap["texto_padronizado"].str.split().str.len().fillna(0).astype(int)

# Datas e duração
sap["data_base"] = to_serial_excel(sap["data_base"])
sap["inicio"] = to_serial_excel(sap["_d_ini"]) + pd.to_timedelta(sap["_h_ini"])
sap["fim"] = to_serial_excel(sap["_d_fim"]) + pd.to_timedelta(sap["_h_fim"])   # NaT se não há fim
sap["horas"] = ((sap["fim"] - sap["inicio"]).dt.total_seconds() / 3600).round(4)

# Máquina pela localização técnica + tabela de-para
niveis = sap["local_instalacao"].str.split(".", expand=True)
sap["cod_maquina_sap"] = niveis[NIVEL_MAQUINA] if NIVEL_MAQUINA in niveis.columns else pd.NA
sap["subconjunto_local"] = niveis[NIVEL_SUBCONJUNTO] if NIVEL_SUBCONJUNTO in niveis.columns else pd.NA
sap["maquina"] = sap["cod_maquina_sap"].map(MAPA_MAQUINA_SAP).astype("string")

sem_mapa = sorted(sap.loc[sap["maquina"].isna() & sap["cod_maquina_sap"].notna(),
                          "cod_maquina_sap"].unique())
if sem_mapa:
    warnings.warn(f"Códigos de máquina sem correspondência em MAPA_MAQUINA_SAP: {sem_mapa}")
    
print(sap.info())
sap.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1552 entries, 0 to 1551
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   n_ordem            1552 non-null   int64         
 1   tipo               1552 non-null   string        
 2   data_base          1552 non-null   datetime64[ns]
 3   texto_original     1552 non-null   string        
 4   local_instalacao   1552 non-null   string        
 5   descricao_equip    1552 non-null   string        
 6   _d_ini             1552 non-null   int64         
 7   _h_ini             1552 non-null   object        
 8   _d_fim             1549 non-null   float64       
 9   _h_fim             1552 non-null   object        
 10  tipo_atividade     1552 non-null   string        
 11  texto_padronizado  1552 non-null   string        
 12  n_palavras         1552 non-null   int64         
 13  inicio             1552 non-null   datetime64[ns]
 14  fim     

,n_ordem,tipo,data_base,texto_original,local_instalacao,descricao_equip,_d_ini,_h_ini,_d_fim,_h_fim,tipo_atividade,texto_padronizado,n_palavras,inicio,fim,horas,cod_maquina_sap,subconjunto_local,maquina
0,1001695816,CORRETIVA,2026-01-04,PROBLEMA NO QMD,9001.FABX.CEL01.MAQA.FRN01,FORNO 1,46028,14:00:00,46028.0,18:00:00,CO,PROBLEMA NO QMD,3,2026-01-06 14:00:00,2026-01-06 18:00:00,4.0000,MAQA,FRN01,A
1,1001696298,CORRETIVA PLANEJADA,2026-01-04,QMD 02 E 05 E A TRANSFERENCIA ENTUPIDA.,9001.FABX.CEL01.MAQA.FRN01,FORNO 1,46026,11:35:00,46026.0,13:20:00,CP,QMD 02 E 05 E A TRANSFERENCIA ENTUPIDA,8,2026-01-04 11:35:00,2026-01-04 13:20:00,1.7500,MAQA,FRN01,A
2,1001696810,CORRETIVA PLANEJADA,2026-01-05,SUBSTITUIR TRAFO DE IGNIÇÃO,9001.FABX.CEL01.MAQA.FRN02,FORNO 2,46027,09:00:00,46027.0,12:15:00,SD,SUBSTITUIR TRAFO DE IGNICAO,4,2026-01-05 09:00:00,2026-01-05 12:15:00,3.2500,MAQA,FRN02,A
3,1001697104,CORRETIVA PLANEJADA,2026-01-06,ALINHAR LONA PÓS DISCO,9001.FABX.CEL01.MAQA.LON01,LONAS,46028,18:05:00,46028.0,19:10:00,CP,ALINHAR LONA POS DISCO,4,2026-01-06 18:05:00,2026-01-06 19:10:00,1.0833,MAQA,LON01,A
4,1001697806,CORRETIVA,2026-01-06,CURTO NA FIAÇÃO DA LONA,9001.FABX.CEL01.MAQA.LON04,LONAS,46028,06:00:00,46028.0,06:20:00,CO,CURTO NA FIACAO DA LONA,5,2026-01-06 06:00:00,2026-01-06 06:20:00,0.3333,MAQA,LON04,A


### Validade e alertas por ordem do SAP

| Regra | Efeito | Por quê, em palavras simples |
|---|---|---|
| número de ordem repetido | **inválido** | É a mesma ordem duas vezes na planilha (cópia exata). |
| sem início; duração zero ou negativa | **inválido** | Tempo impossível. |
| mesma localização + mesmo texto **com horários sobrepostos** (ou mesmo início) | **inválido** (duplicata provável) | A mesma intervenção lançada duas vezes: mesmo lugar, mesma descrição, mesmo horário. |
| texto vazio | **inválido** | Sem texto não dá para saber o que foi feito, e a classificação dos textos (Subproduto 3) não consegue usar a ordem. |
| mesma localização + mesmo texto no mesmo dia, **sem** sobreposição de horário | alerta | Pode ser **reincidência real** (o problema voltou à tarde). Como os textos são curtos ("FALHA NO QMD"), texto igual não basta para afirmar que é duplicata. |
| sem data de fim | alerta | A ordem pode ter ficado aberta. Ela conta como evento, mas sem duração. |
| acima de `DURACAO_MAX_H` = 24 h | alerta | É tempo de **calendário** (da abertura ao fechamento da ordem), que pode incluir noites e esperas. Não é tempo de mão na máquina. |

**Por que a duplicata exige horários sobrepostos:** "TROCAR LONA" duas vezes no mesmo dia pode ser duas trocas de verdade (a lona rasgou de novo). Só quando os horários se **encostam ou se cruzam** é que fica claro que é o mesmo serviço digitado duas vezes. Na dúvida, o registro recebe um alerta, não uma exclusão.

In [17]:
DURACAO_MAX_H = 24

sap = sap.drop(columns=["valido", "motivo_invalido", "alertas",
                        "confiavel", "motivo_desconfianca", "provavel_duplicata"], errors="ignore")

invalido = pd.Series("", index=sap.index)
alerta = pd.Series("", index=sap.index)

# ---- Inválidos ----
invalido = add_motivo(invalido, sap["n_ordem"].duplicated(keep="first"), "n_ordem repetido")
invalido = add_motivo(invalido, sap["inicio"].isna(), "sem início")
invalido = add_motivo(invalido, sap["horas"] <= 0, "duração zero ou negativa")

# Duplicata provável: mesma localização + mesmo texto e horários que se sobrepõem (ou mesmo início)
chave = ["local_instalacao", "texto_padronizado"]
o = (sap.loc[sap["inicio"].notna() & sap["texto_padronizado"].notna()]
        .sort_values(chave + ["inicio", "fim"]))
inicio_ant = o.groupby(chave)["inicio"].shift()
dup = (o["inicio"] < fim_max_anterior(o, chave)) | (o["inicio"] == inicio_ant)
dup = dup.reindex(sap.index, fill_value=False)
invalido = add_motivo(invalido, dup, "duplicata provável: mesmo local e texto com horário sobreposto")
invalido = add_motivo(invalido, sap["texto_padronizado"].isna(), "texto vazio")

# ---- Alertas ----
mesmo_dia = (sap["texto_padronizado"].notna()
             & sap.duplicated(["local_instalacao", "texto_padronizado", "_d_ini"], keep=False)
             & ~dup)
alerta = add_motivo(alerta, mesmo_dia, "mesmo local e texto no mesmo dia (possível reincidência)")
alerta = add_motivo(alerta, sap["fim"].isna(), "sem data de fim (ordem possivelmente aberta)")
alerta = add_motivo(alerta, sap["horas"] > DURACAO_MAX_H,
                    f"acima de {DURACAO_MAX_H} h de calendário (pode incluir pausas)")


sap["valido"] = invalido.eq("")
sap["motivo_invalido"] = invalido
sap["alertas"] = alerta

sap = sap[[
    "n_ordem",
    "maquina",
    "cod_maquina_sap",
    "descricao_equip",
    "local_instalacao",
    "subconjunto_local",
    "tipo",
    "tipo_atividade",
    "texto_original",
    "texto_padronizado",
    "n_palavras",
    "data_base",
    "inicio",
    "fim",
    "horas",
    "valido",
    "motivo_invalido",
    "alertas",
]]

print(sap["valido"].value_counts())
print(sap.loc[~sap["valido"], "motivo_invalido"].value_counts())
print(f"total de descartados: {sap.loc[~sap['valido'], 'n_ordem'].nunique()} ordens")
print(f"total de ordens rotuladas com alerta: {sap.loc[sap['alertas'] != '', 'n_ordem'].nunique()}")
sap.head()

valido
True     1547
False       5
Name: count, dtype: int64
motivo_invalido
duplicata provável: mesmo local e texto com horário sobreposto    4
duração zero ou negativa                                          1
Name: count, dtype: int64
total de descartados: 5 ordens
total de ordens rotuladas com alerta: 33


,n_ordem,maquina,cod_maquina_sap,descricao_equip,local_instalacao,subconjunto_local,tipo,tipo_atividade,texto_original,texto_padronizado,n_palavras,data_base,inicio,fim,horas,valido,motivo_invalido,alertas
0,1001695816,A,MAQA,FORNO 1,9001.FABX.CEL01.MAQA.FRN01,FRN01,CORRETIVA,CO,PROBLEMA NO QMD,PROBLEMA NO QMD,3,2026-01-04,2026-01-06 14:00:00,2026-01-06 18:00:00,4.0000,True,,
1,1001696298,A,MAQA,FORNO 1,9001.FABX.CEL01.MAQA.FRN01,FRN01,CORRETIVA PLANEJADA,CP,QMD 02 E 05 E A TRANSFERENCIA ENTUPIDA.,QMD 02 E 05 E A TRANSFERENCIA ENTUPIDA,8,2026-01-04,2026-01-04 11:35:00,2026-01-04 13:20:00,1.7500,True,,
2,1001696810,A,MAQA,FORNO 2,9001.FABX.CEL01.MAQA.FRN02,FRN02,CORRETIVA PLANEJADA,SD,SUBSTITUIR TRAFO DE IGNIÇÃO,SUBSTITUIR TRAFO DE IGNICAO,4,2026-01-05,2026-01-05 09:00:00,2026-01-05 12:15:00,3.2500,True,,
3,1001697104,A,MAQA,LONAS,9001.FABX.CEL01.MAQA.LON01,LON01,CORRETIVA PLANEJADA,CP,ALINHAR LONA PÓS DISCO,ALINHAR LONA POS DISCO,4,2026-01-06,2026-01-06 18:05:00,2026-01-06 19:10:00,1.0833,True,,
4,1001697806,A,MAQA,LONAS,9001.FABX.CEL01.MAQA.LON04,LON04,CORRETIVA,CO,CURTO NA FIAÇÃO DA LONA,CURTO NA FIACAO DA LONA,5,2026-01-06,2026-01-06 06:00:00,2026-01-06 06:20:00,0.3333,True,,


**O que a saída mostra:** das 1.552 ordens, só 5 foram invalidadas (4 duplicatas prováveis e 1 com duração impossível), e 33 receberam algum alerta. Assim como no MES, a base é aproveitável quase inteira.

# Exportação dos dados tratados

As três tabelas limpas são salvas para os próximos notebooks:

- `mes_registros`: uma linha por registro do MES, já com `id_parada`, `valido` e `alertas`;
- `mes_paradas`: uma linha por parada (o tempo parado já calculado pela união);
- `sap_ordens`: uma linha por ordem do SAP, com `valido` e `alertas`.

**Por que o formato `parquet`:** é um formato de arquivo feito para dados em tabela. Diferente do Excel ou do CSV, ele **guarda o tipo de cada coluna** (data continua data, número continua número), é menor e mais rápido de ler. Uma cópia em Excel também é gerada, mas **só para olhar** à mão, não para alimentar a próxima etapa.

**Por que salvar os parâmetros (`parametros.json`):** é a "receita" usada nesta rodada (a folga de 2 min, o limite de micro-parada, o limite de 24 h, a tabela de-para). Se um número mudar no futuro, dá para saber se foi porque os dados mudaram ou porque a receita mudou.

In [18]:
from datetime import datetime
import importlib.util
import json
from pathlib import Path

OUT_DIR = Path("./data/processed/limpeza").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

saidas = {
    "mes_registros": mes,      # 1 linha por registro do MES (com id_parada, valido, alertas)
    "mes_paradas":   paradas,  # 1 linha por parada (duracao_h = união dos intervalos)
    "sap_ordens":    sap,      # 1 linha por ordem do SAP (com valido, alertas)
}

FORMAT = "parquet"

for nome, df in saidas.items():
    caminho = OUT_DIR / f"{nome}.{FORMAT}"
    df.to_parquet(caminho, index=False)

# Cópia em Excel só para inspeção manual (não use como entrada da próxima etapa)
with pd.ExcelWriter(OUT_DIR / "base_tratada_inspecao.xlsx") as xw:
    for nome, df in saidas.items():
        df.to_excel(xw, sheet_name=nome, index=False)

# Parâmetros usados, para saber de que versão do tratamento cada arquivo saiu
parametros = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "arquivo_origem": str(FILE),
    "formato": FORMAT,
    "GAP_MAX_MIN": GAP_MAX_MIN,
    "MICRO_PARADA_MIN": MICRO_PARADA_MIN,
    "DURACAO_MAX_H": DURACAO_MAX_H,
    "MAPA_MAQUINA_SAP": MAPA_MAQUINA_SAP,
    "linhas": {nome: len(df) for nome, df in saidas.items()},
}
(OUT_DIR / "parametros.json").write_text(json.dumps(parametros, ensure_ascii=False, indent=2), encoding="utf-8")

for nome, df in saidas.items():
    print(f"{nome:15s} {len(df):>6} linhas -> {OUT_DIR / (nome + '.' + FORMAT)}")

mes_registros     1640 linhas -> C:\Users\girle\Desktop\Yan\1. UFPE\2026.2\COMPET - - MOURA\moura_manutencao\src\data\processed\limpeza\mes_registros.parquet
mes_paradas       1501 linhas -> C:\Users\girle\Desktop\Yan\1. UFPE\2026.2\COMPET - - MOURA\moura_manutencao\src\data\processed\limpeza\mes_paradas.parquet
sap_ordens        1552 linhas -> C:\Users\girle\Desktop\Yan\1. UFPE\2026.2\COMPET - - MOURA\moura_manutencao\src\data\processed\limpeza\sap_ordens.parquet
